# HW#8 - Part 1: Two-Layer Neural Network with MNIST (Keras)

**Author:** Nabonita Das
**Course:** ECE-5831

*Works in Google Colab and locally. MNIST downloads automatically, so no upload is needed.*

**Goal:** Implement a `TwoLayerNetKeras` class using Keras, train it on the MNIST training set, test it on the MNIST test set, and verify that training and testing work properly.

**Architecture:** `Input(784) -> Dense(hidden, ReLU) -> Dense(10, Softmax)`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
tf.config.set_visible_devices([], "GPU")   # CPU-only version: ignore any GPU
from tensorflow import keras
from tensorflow.keras import layers, regularizers

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("Keras     :", keras.__version__)

## 1. Load the MNIST dataset (Keras dataset library)

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

print("x_train:", x_train.shape, x_train.dtype, "| y_train:", y_train.shape)
print("x_test :", x_test.shape, x_test.dtype, "| y_test :", y_test.shape)
print("Classes:", np.unique(y_train))

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(x_train[i], cmap="gray")
    ax.set_title(f"label: {y_train[i]}")
    ax.axis("off")
plt.suptitle("Sample MNIST training images")
plt.tight_layout()
plt.show()

## 2. Pre-processing

* Flatten each 28x28 image into a 784-dimensional vector.
* Scale pixel values from [0, 255] to [0, 1].
* Hold out 10% of the training set as a validation set so we can monitor over-fitting while training.
* The **test set is never used for training or model selection.**

In [ ]:
NUM_CLASSES = 10
INPUT_DIM = 28 * 28

x_train_f = (x_train.reshape(-1, INPUT_DIM) / 255.0).astype("float32")
x_test_f = (x_test.reshape(-1, INPUT_DIM) / 255.0).astype("float32")

# hold out the last 10% of the training data for validation
n_val = int(0.1 * len(x_train_f))
x_val, y_val = x_train_f[-n_val:], y_train[-n_val:]
x_tr, y_tr = x_train_f[:-n_val], y_train[:-n_val]

print("train     :", x_tr.shape, y_tr.shape)
print("validation:", x_val.shape, y_val.shape)
print("test      :", x_test_f.shape, y_test.shape)
print("pixel range:", x_tr.min(), "-", x_tr.max())

## 3. The `TwoLayerNetKeras` class

The class wraps a Keras `Sequential` model with one hidden layer and one output layer, and exposes `train`, `predict`, `evaluate` and `summary` methods (mirroring the two-layer net written from scratch earlier in the course).

In [ ]:
class TwoLayerNetKeras:
    """Two-layer fully connected neural network implemented with Keras.

    input -> Dense(hidden_size, ReLU) -> Dense(output_size, Softmax)
    """

    def __init__(self, input_size=784, hidden_size=128, output_size=10,
                 learning_rate=1e-3, weight_decay=0.0):
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size

        reg = regularizers.l2(weight_decay) if weight_decay > 0 else None
        self.model = keras.Sequential([
            keras.Input(shape=(input_size,)),
            layers.Dense(hidden_size, activation="relu",
                         kernel_regularizer=reg, name="hidden"),
            layers.Dense(output_size, activation="softmax", name="output"),
        ], name="two_layer_net")

        self.model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
            loss="sparse_categorical_crossentropy",
            metrics=["accuracy"],
        )

    def train(self, x, y, epochs=10, batch_size=128, validation_data=None, verbose=1):
        """Train the network and return the Keras History object."""
        return self.model.fit(x, y, epochs=epochs, batch_size=batch_size,
                              validation_data=validation_data, verbose=verbose)

    def predict(self, x):
        """Return class probabilities, shape (N, output_size)."""
        return self.model.predict(x, verbose=0)

    def predict_classes(self, x):
        """Return predicted class labels, shape (N,)."""
        return np.argmax(self.predict(x), axis=1)

    def evaluate(self, x, y):
        """Return (loss, accuracy)."""
        return self.model.evaluate(x, y, verbose=0)

    def summary(self):
        self.model.summary()


net = TwoLayerNetKeras(input_size=INPUT_DIM, hidden_size=128, output_size=NUM_CLASSES,
                       learning_rate=1e-3)
net.summary()

## 4. Sanity checks before full training

Two quick checks that the pipeline is wired correctly:

1. **Initial loss:** with random weights and 10 balanced classes the cross-entropy loss should be close to `ln(10) = 2.303`.
2. **Over-fit a tiny subset:** the network should be able to memorise 50 samples (accuracy -> ~100%). If it cannot, there is a bug in the training code.

In [ ]:
# Check 1: initial loss
init_loss, init_acc = net.evaluate(x_tr[:2000], y_tr[:2000])
print(f"Initial loss     : {init_loss:.4f}   (expected about ln(10) = {np.log(10):.4f})")
print(f"Initial accuracy : {init_acc:.4f}   (expected about 0.10)")

# Check 2: over-fit 50 samples with a fresh network
tiny_net = TwoLayerNetKeras(INPUT_DIM, 128, NUM_CLASSES, learning_rate=1e-3)
h = tiny_net.train(x_tr[:50], y_tr[:50], epochs=100, batch_size=50, verbose=0)
print(f"Tiny-subset final train accuracy: {h.history['accuracy'][-1]:.4f}")
assert h.history["accuracy"][-1] > 0.95, "Network failed to over-fit a tiny subset"
print("Sanity checks passed.")

## 5. Train on the MNIST training dataset

In [ ]:
EPOCHS = 10
BATCH_SIZE = 128

history = net.train(x_tr, y_tr, epochs=EPOCHS, batch_size=BATCH_SIZE,
                    validation_data=(x_val, y_val))

### Training curves

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(history.history["loss"], label="train")
ax[0].plot(history.history["val_loss"], label="validation")
ax[0].set_title("Loss"); ax[0].set_xlabel("epoch"); ax[0].legend(); ax[0].grid(alpha=0.3)

ax[1].plot(history.history["accuracy"], label="train")
ax[1].plot(history.history["val_accuracy"], label="validation")
ax[1].set_title("Accuracy"); ax[1].set_xlabel("epoch"); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Test on the MNIST test dataset

In [ ]:
train_loss, train_acc = net.evaluate(x_tr, y_tr)
val_loss, val_acc = net.evaluate(x_val, y_val)
test_loss, test_acc = net.evaluate(x_test_f, y_test)

print(f"Train      -> loss: {train_loss:.4f}  accuracy: {train_acc:.4f}")
print(f"Validation -> loss: {val_loss:.4f}  accuracy: {val_acc:.4f}")
print(f"Test       -> loss: {test_loss:.4f}  accuracy: {test_acc:.4f}")

### Confusion matrix and per-class accuracy on the test set

In [ ]:
y_pred = net.predict_classes(x_test_f)
cm = tf.math.confusion_matrix(y_test, y_pred, num_classes=NUM_CLASSES).numpy()

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(10)); ax.set_yticks(range(10))
ax.set_xlabel("Predicted label"); ax.set_ylabel("True label")
ax.set_title("Confusion matrix (MNIST test set)")
thresh = cm.max() / 2
for i in range(10):
    for j in range(10):
        ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=8,
                color="white" if cm[i, j] > thresh else "black")
plt.colorbar(im)
plt.tight_layout()
plt.show()

per_class = cm.diagonal() / cm.sum(axis=1)
for digit, acc in enumerate(per_class):
    print(f"digit {digit}: accuracy = {acc:.4f}")

### Example predictions (green = correct, red = wrong)

In [ ]:
rng = np.random.default_rng(SEED)
idx = rng.choice(len(x_test), 16, replace=False)

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for ax, i in zip(axes.ravel(), idx):
    ax.imshow(x_test[i], cmap="gray")
    ok = y_pred[i] == y_test[i]
    ax.set_title(f"pred {y_pred[i]} / true {y_test[i]}", color="green" if ok else "red", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

wrong = np.where(y_pred != y_test)[0]
print(f"Misclassified test images: {len(wrong)} of {len(y_test)}")
fig, axes = plt.subplots(1, 8, figsize=(14, 2.5))
for ax, i in zip(axes, wrong[:8]):
    ax.imshow(x_test[i], cmap="gray")
    ax.set_title(f"pred {y_pred[i]} / true {y_test[i]}", color="red", fontsize=9)
    ax.axis("off")
plt.suptitle("Some misclassified test images")
plt.tight_layout()
plt.show()

## 7. Verify that training and testing work properly

In [ ]:
# 1) training loss decreased and training accuracy increased
assert history.history["loss"][-1] < history.history["loss"][0], "Loss did not decrease"
assert history.history["accuracy"][-1] > history.history["accuracy"][0], "Accuracy did not increase"

# 2) the trained model is far better than chance (10%)
assert test_acc > 0.95, f"Test accuracy too low: {test_acc:.4f}"

# 3) Keras' accuracy matches a manual computation from the predictions
manual_acc = np.mean(y_pred == y_test)
assert abs(manual_acc - test_acc) < 1e-4, "Manual and Keras accuracy disagree"

# 4) predicted probabilities are valid distributions
probs = net.predict(x_test_f[:100])
assert np.allclose(probs.sum(axis=1), 1.0, atol=1e-4)

print("All verification checks passed.")
print(f"Final test accuracy: {test_acc * 100:.2f}%  (manual check: {manual_acc * 100:.2f}%)")

## 8. Per-class precision, recall and F1 (test set)

In [ ]:
print(f"{'digit':>5} {'precision':>10} {'recall':>8} {'F1':>8} {'support':>8}")
f1s = []
for d in range(NUM_CLASSES):
    tp = cm[d, d]; fp = cm[:, d].sum() - tp; fn = cm[d, :].sum() - tp
    prec = tp / max(tp + fp, 1); rec = tp / max(tp + fn, 1)
    f1 = 2 * prec * rec / max(prec + rec, 1e-12); f1s.append(f1)
    print(f"{d:>5} {prec:>10.4f} {rec:>8.4f} {f1:>8.4f} {cm[d].sum():>8}")
print(f"\nmacro F1: {np.mean(f1s):.4f}")

## 9. Extra experiment: effect of the hidden-layer size

The same `TwoLayerNetKeras` class is trained for 5 epochs with different numbers of hidden units to see how the size of the hidden layer affects accuracy.

In [ ]:
sizes = [16, 64, 128, 256]
exp = {}
for hs in sizes:
    tmp = TwoLayerNetKeras(INPUT_DIM, hs, NUM_CLASSES, learning_rate=1e-3)
    tmp.train(x_tr, y_tr, epochs=5, batch_size=BATCH_SIZE, validation_data=(x_val, y_val), verbose=0)
    exp[hs] = tmp.evaluate(x_test_f, y_test)
    print(f"hidden units = {hs:>3}  ->  test loss {exp[hs][0]:.4f} | test accuracy {exp[hs][1]:.4f}")

plt.figure(figsize=(5, 3.5))
plt.plot(sizes, [exp[h][1] for h in sizes], marker="o")
plt.xlabel("hidden units"); plt.ylabel("test accuracy"); plt.title("Hidden-layer size vs. accuracy")
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 10. Save the trained model and check that it reloads correctly

In [ ]:
import os, json
os.makedirs("results", exist_ok=True)
net.model.save("results/two_layer_net_mnist.keras")

reloaded = keras.models.load_model("results/two_layer_net_mnist.keras")
same = np.allclose(reloaded.predict(x_test_f[:500], verbose=0), net.predict(x_test_f[:500]), atol=1e-5)
assert same, "Reloaded model gives different predictions"
print("Model saved to results/two_layer_net_mnist.keras and reloaded with identical predictions.")

with open("results/two_layer_net_results.json", "w") as f:
    json.dump({"author": "Nabonita Das", "epochs": EPOCHS, "batch_size": BATCH_SIZE,
               "train_accuracy": float(train_acc), "validation_accuracy": float(val_acc),
               "test_accuracy": float(test_acc), "test_loss": float(test_loss),
               "macro_f1": float(np.mean(f1s))}, f, indent=2)

## 11. Conclusion

A two-layer fully connected network built with Keras (`TwoLayerNetKeras`) was trained on the MNIST training set and evaluated on the MNIST test set.

* The initial loss was close to `ln(10)`, and the network could over-fit a 50-sample subset, confirming the training code is correct.
* The training and validation loss decreased and the accuracy increased over the 10 epochs (see the curves above).
* The test accuracy is far above chance (10%) and agrees with a manual accuracy computation, and the confusion matrix and example predictions show the network is classifying digits correctly.

These results verify that both the training and testing processes work properly.